# 0. Environment setting

## Libary import

In [2]:
import pandas as pd
from datetime import date, timedelta
import re
import requests
import json
import urllib3
from difflib import get_close_matches
import ctypes
import threading
import time
import os

## Qualtrics Credentials

In [3]:
# ============================================================
# CONFIGURATION 
# ============================================================

API_TOKEN = "dZZueEgbaPrCuSTXEtIp2sz0EqlJNxg93jYEod7U"   # <-- insert your token
DATA_CENTER = "iad1"
SURVEY_ID = "SV_efytbaOtuAX2uF0"

## Success/failure message

In [4]:
def popup_info(message, title="Success", timeout=10):
    MB_OK = 0x0
    MB_ICONINFORMATION = 0x40

    def show_box():
        ctypes.windll.user32.MessageBoxW(0, message, title, MB_OK | MB_ICONINFORMATION)

    t = threading.Thread(target=show_box)
    t.start()

    time.sleep(timeout)

    hwnd = ctypes.windll.user32.FindWindowW(None, title)
    if hwnd:
        ctypes.windll.user32.PostMessageW(hwnd, 0x0010, 0, 0)  # WM_CLOSE


def popup_error(message, title="Error", timeout=10):
    MB_OK = 0x0
    MB_ICONERROR = 0x10

    def show_box():
        ctypes.windll.user32.MessageBoxW(0, message, title, MB_OK | MB_ICONERROR)

    t = threading.Thread(target=show_box)
    t.start()

    time.sleep(timeout)

    hwnd = ctypes.windll.user32.FindWindowW(None, title)
    if hwnd:
        ctypes.windll.user32.PostMessageW(hwnd, 0x0010, 0, 0)


## Get sharepoint landing folder

In [5]:
def get_onedrive_path():
    return os.path.join(os.path.expanduser("~"), "OneDrive - IBERDROLA S.A")

def get_sharepoint_folder():
    onedrive_root = get_onedrive_path()
    return os.path.join(
        onedrive_root,
        "General - Customer Research",
        "Post Call Survey",
        "Post Call Survey Data 2025",
        "Avangrid_NY"
    )

def get_output_path(filename):
    sharepoint_folder = get_sharepoint_folder()

    if os.path.exists(sharepoint_folder):
        return os.path.join(sharepoint_folder, filename), True

    return filename, False


# 1. Extract

## Initial data extraction
Task:
- Get all theraw data from the file despite the format
- Get the column names from the file context (If the order changes the dataframe keeps it consistent)

In [6]:

def extract_data(input_path):

    # Read the full sheet (no assumptions about columns)
    raw = pd.read_excel(input_path, header=None, dtype=str)

    # Row 7 contains the question text
    question_row = raw.iloc[6].fillna("").astype(str)

    # Data starts at row 9
    data = raw.iloc[8:].reset_index(drop=True)

    # Prepare final column names list
    final_cols = []

    for col_idx, col_series in data.items():
        sample_value = col_series.dropna().astype(str).iloc[0] if col_series.dropna().size > 0 else ""
        question_text = question_row[col_idx].lower()

        # -------------------------
        # METADATA COLUMN DETECTION
        # -------------------------

        # ID
        if re.match(r"^[UE]\d+", sample_value):
            final_cols.append("ID")
            continue

        # Name (column immediately right of ID)
        if len(final_cols) > 0 and final_cols[-1] == "ID":
            final_cols.append("Name")
            continue

       # Date/Time (column immediately right of Name)
        if len(final_cols) > 0 and final_cols[-1] == "Name":
            final_cols.append("Date/Time")
            continue

        # InteractionID (alphanumeric)
        if re.match(r"^[A-Za-z0-9]{12,}$", sample_value) and not sample_value.startswith("+"):
            final_cols.append("InteractionID")
            continue

        # Phone Number
        if sample_value.startswith("+"):
            final_cols.append("Phone Number")
            continue

        # Survey Name
        if any(x in sample_value.lower() for x in ["survey", "surv"]):
            final_cols.append("Survey Name")
            continue

        # Work Group
        if any(x in sample_value.lower() for x in ["cc", "vendor", "new"]):
            final_cols.append("Work Group")
            continue

        # -------------------------
        # SCORING COLUMN DETECTION
        # -------------------------

        qt = question_text  # shorthand

        if "recommend" in qt:
            final_cols.append("NPS")
            continue

        if "resolve" in qt or "call back" in qt:
            final_cols.append("FCR")
            continue

        if "help" in qt:
            final_cols.append("E_H")
            continue

        if any(x in qt for x in ["clear", "explain", "explaine"]):
            final_cols.append("C_E")
            continue

        if "satisfied" in qt:
            final_cols.append("CSAT")
            continue

        if any(x in qt for x in ["payment", "billing", "outage"]):
            final_cols.append("Call Reason")
            continue

        # If nothing matches, mark as Unknown
        final_cols.append(f"Unknown_{col_idx}")

    # Apply the detected column names
    data.columns = final_cols

    # Convert Date/Time to proper format
    if "Date/Time" in data.columns:
        data["Date/Time"] = pd.to_datetime(data["Date/Time"], errors="coerce")
        data["Date/Time"] = data["Date/Time"].dt.strftime("%m/%d/%Y %H:%M:%S")
    
    # Convert scoring columns to integers
    score_cols = ["NPS", "FCR", "E_H", "C_E", "CSAT", "Call Reason"]

    for col in score_cols:
        if col in data.columns:
            data[col] = pd.to_numeric(data[col], errors="coerce").astype("Int64")

    
    #display(data.head(5))

    return data



# 2. Transform

## Add metadata columns (Survey Status/Completion & Tag)
Tasks:
- Delete empty columns
- Compute Survey Completion and Tag Fields
- Reorder the fields matching the Qualtrics survey order

In [7]:
def transform_data(df):

    # 0. Remove empty columns
    df = df.dropna(axis=1, how="all")

    # 1. Detect unknown columns
    expected_cols = ["ID", "Name", "Date/Time", "InteractionID", "Phone Number",
                     "Survey Name", "Work Group", "NPS", "FCR", "E_H",
                     "C_E", "CSAT", "Call Reason"]

    unknown_cols = [c for c in df.columns if c not in expected_cols]

    # Windows popup warning
    if unknown_cols:
        import ctypes
        message = f"Unknown columns detected:\n{unknown_cols}"
        ctypes.windll.user32.MessageBoxW(0, message, "ETL Warning", 0x40)

    # 2. Validate scoring columns
    required_cols = ["NPS", "FCR", "E_H", "C_E", "CSAT", "Call Reason"]
    missing = [col for col in required_cols if col not in df.columns]
    if missing:
        raise ValueError(f"Missing required scoring columns: {missing}")

    # 3. Validate Work Group and CSAT exist before popping
    if "Work Group" not in df.columns:
        raise ValueError("Column 'Work Group' is missing from the dataset.")

    if "CSAT" not in df.columns:
        raise ValueError("Column 'CSAT' is missing from the dataset.")

    # Move Columns
    csat = df.pop("CSAT")
    work_group = df.pop("Work Group")

    df.insert(3, "Work Group", work_group)
    df.insert(7, "CSAT", csat)

    # Null handling of Name and ID
    df[["ID", "Name"]] = df[["ID", "Name"]].ffill()

    # Create Survey Status/Completion column
    df["Survey Status"] = df[required_cols].notna().all(axis=1)
    df["Survey Status"] = df["Survey Status"].map({True: "Complete", False: "Abandoned"})

    # Tag column
    df["Tag"] = df["Work Group"].str.contains("test", case=True, na=False).map({True: "Test", False: ""})

    # Convert scoring fields into integers
    for col in required_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce").astype("Int64")

    #display(df.head(5))
    return df


## Prepare format for Qualtrics
Tasks:
- Get real field names form the qualtrics targeted survey
- Map dataframe field with official survey field names
- Add extra metadata rows for qualtrics understanding

In [8]:
def prepare_for_qualtrics(df):

    urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

    url = f"https://{DATA_CENTER}.qualtrics.com/API/v3/surveys/{SURVEY_ID}"

    headers = {
        "X-API-TOKEN": API_TOKEN
    }

    response = requests.get(url, headers=headers, verify=False)
    
    # Convert API response to JSON
    survey_json = response.json()

    # Pull the Qualtrics question metadata
    label_map = survey_json["result"]["questions"]

    # QID → questionName / questionText
    qid_to_name = {qid: q["questionName"] for qid, q in label_map.items()}
    qid_to_text = {qid: q["questionText"] for qid, q in label_map.items()}

    # questionName → QID
    name_to_qid = {name: qid for qid, name in qid_to_name.items()}

    # All official Qualtrics labels (questionName)
    qualtrics_names = list(name_to_qid.keys())

    # 1) Fuzzy‑match df columns to Qualtrics questionName
    mapped_cols = {}
    used_names = set()

    for col in df.columns:
        match = get_close_matches(col, qualtrics_names, n=1, cutoff=0.6)
        if match:
            new_name = match[0]
            if new_name in used_names:
                new_name = col  # avoid duplicates
            mapped_cols[col] = new_name
            used_names.add(new_name)
        else:
            mapped_cols[col] = col  # leave unmapped as‑is

    df = df.rename(columns=mapped_cols)

    # 2) Row 2: questionText (aligned to final column names)
    row2 = []
    for col in df.columns:
        qid = name_to_qid.get(col)
        row2.append(qid_to_text.get(qid, "") if qid else "")

    # 3) Row 3: {"ImportId": "QIDx_TEXT"} as STRING
    row3 = []
    for col in df.columns:
        qid = name_to_qid.get(col)
        if qid:
            row3.append(f'{{"ImportId": "{qid}_TEXT"}}')
        else:
            row3.append("")

    # 4) Stack as extra rows (no new columns)
    # 4) Stack as extra rows (NO new columns)
    hdr1 = pd.DataFrame([list(df.columns)], columns=df.columns)  # row 1: questionName
    hdr2 = pd.DataFrame([row2], columns=df.columns)              # row 2: questionText
    hdr3 = pd.DataFrame([row3], columns=df.columns)              # row 3: ImportId/QID_TEXT

    df_final = pd.concat([hdr1, hdr2, hdr3, df.reset_index(drop=True)], ignore_index=True)

    # Remove duplicated header row and reset index
    df_final = df_final.iloc[1:].reset_index(drop=True)

   
    display(df_final.head(5))
    return df_final


# 3. Load

## File csv creation after transformation
Tasks:
- Create repository file
- Create Queatrics ready temporary file

In [9]:
def load_data(df, output_path):
    df.to_csv(output_path, index=False, encoding="utf-8")

    return


## Source folder mapping

In [10]:
# Extract Date fields
today = date.today()

# If today is Monday (weekday() == 0), use last Saturday
if today.weekday() == 0:
    effective_date = today - timedelta(days=2)
else:
    effective_date = today

# Extract Date fields
source_year = effective_date.year
source_month = effective_date.strftime("%B")
source_day = effective_date.strftime("%d")
source_weekday = effective_date.weekday()
landing_yesterday = effective_date - timedelta(days=1)

#print(source_year, source_month, source_day, source_weekday, landing_yesterday)


## Post to Qualtrics

In [11]:
def upload_to_qualtrics(file_path, DATA_CENTER, SURVEY_ID, API_TOKEN):
    """
    Uploads a CSV file to Qualtrics using the Import Responses API.
    Expects a fully formatted Qualtrics-ready CSV at file_path.
    """

    urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

    url = f"https://{DATA_CENTER}.qualtrics.com/API/v3/surveys/{SURVEY_ID}/import-responses"

    headers = {
        "X-API-TOKEN": API_TOKEN,
        "Content-Type": "text/csv",
        "charset": "UTF-8"
    }

    print(f"\nUploading file to Qualtrics: {file_path}")

    with open(file_path, "rb") as f:
        response = requests.post(
            url,
            headers=headers,
            data=f,
            verify=False
        )

    print("\n=== RAW RESPONSE TEXT ===")
    print(response.text)
    print("=========================\n")

    try:
        result = response.json()
        print("Upload response (parsed):")
        print(json.dumps(result, indent=4))
        return result
    except Exception:
        print("Could not parse JSON response.")
        return response.text


# Execute

In [12]:
try :
    input_file_path = rf"\\clornas01\DIGITAL_COE_CS_DATA\data_delivery\qualtrics\NY_post_call_survey\daily\{source_year}\{source_month}\{source_day}\NY Feedback Daily.xls"
    #input_file_path = r"\\clornas01\DIGITAL_COE_CS_DATA\data_delivery\qualtrics\NY_post_call_survey\daily\2026\April\03\NY Feedback Daily.xls"
    #output_file_path = r"~\Desktop\NY Feedback Daily Cleaned March 5.csv"
    #output_file_path = "NY_qualtrics_upload6.csv"   # local temp file
    #output_file_path = f"NY Feedback Daily {landing_yesterday}.csv"
    filename = f"NY Feedback Daily {landing_yesterday}.csv"
    output_file_path, used_sharepoint = get_output_path(filename)

    temp_file_path = "NY_qualtrics_upload.csv"   # local temp file
    data = extract_data(input_file_path)
    cleaned_data = transform_data(data)
    load_data(cleaned_data, output_file_path)
    ready_data = prepare_for_qualtrics(cleaned_data)
    load_data(ready_data, temp_file_path)
    result = upload_to_qualtrics(temp_file_path, DATA_CENTER, SURVEY_ID, API_TOKEN)
    
except Exception as e:
    error_message = f"Error running the program:\n{e}"

    if today.weekday() == 0:
        error_message += "\n\nMonday run — Friday folder expected."
    elif today.weekday() == 6:
        error_message += "\n\nWeekend run — folder may be empty."

    popup_error(error_message)

else:
    popup_info(filename,"Upload successful")


C:\Users\E978423\AppData\Local\Temp\ipykernel_20756\848393842.py:40: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[["ID", "Name"]] = df[["ID", "Name"]].ffill()
C:\Users\E978423\AppData\Local\Temp\ipykernel_20756\848393842.py:43: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["Survey Status"] = df[required_cols].notna().all(axis=1)
C:\Users\E978423\AppData\Local\Temp\ipykernel_20756\848393842.py:44: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .lo

,ID,Name,Date/Time,Work Group,InteractionID,Phone Number,Survey Name,CSAT,NPS,FCR,E_H,C_E,Call Reason,Survey Status,Tag
0,ID,Name,Date/Time,Work Group,InteractionID,Phone Number,Survey Name,CSAT,NPS,FCR,Ease of Help,Clear Explanation,Call Reason,Survey Status,Tag
1,"{""ImportId"": ""QID1_TEXT""}","{""ImportId"": ""QID2_TEXT""}","{""ImportId"": ""QID3_TEXT""}","{""ImportId"": ""QID4_TEXT""}","{""ImportId"": ""QID5_TEXT""}","{""ImportId"": ""QID6_TEXT""}","{""ImportId"": ""QID7_TEXT""}","{""ImportId"": ""QID12_TEXT""}","{""ImportId"": ""QID8_TEXT""}","{""ImportId"": ""QID9_TEXT""}","{""ImportId"": ""QID14_TEXT""}","{""ImportId"": ""QID15_TEXT""}","{""ImportId"": ""QID10_TEXT""}","{""ImportId"": ""QID11_TEXT""}","{""ImportId"": ""QID13_TEXT""}"
2,U360453,Erin Judge,04/27/2026 11:42:28,NYS CC MIMO,2002586091D0260427,+15854787256,NYS Survey 20260219,5,9,1,4,5,2,Complete,
3,U339211,Carrie Woolfolk,04/27/2026 12:02:33,NYS CC Vendor Transfer MiMo,2002587263D0260427,+13474957689,NYS Survey 20260219,1,0,0,1,1,6,Complete,
4,U339211,Carrie Woolfolk,04/27/2026 14:49:46,RGE CC Vendor Transfer MiMo,2002615146D0260427,+19739441999,RGE Survey 20260219,5,10,1,5,5,2,Complete,



Uploading file to Qualtrics: NY_qualtrics_upload.csv

=== RAW RESPONSE TEXT ===
{"result":{"progressId":"7bc83236-98b5-4171-ba8b-e8c47f07a091","percentComplete":0.0,"status":"inProgress"},"meta":{"requestId":"a12484e3-a79f-48cf-b79a-311cef292280","httpStatus":"200 - OK"}}

Upload response (parsed):
{
    "result": {
        "progressId": "7bc83236-98b5-4171-ba8b-e8c47f07a091",
        "percentComplete": 0.0,
        "status": "inProgress"
    },
    "meta": {
        "requestId": "a12484e3-a79f-48cf-b79a-311cef292280",
        "httpStatus": "200 - OK"
    }
}
